[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WesamKhallaf73/pytorch-ground-up-notebooks/blob/main/notebooks/L13.ipynb)

# Lecture 13 — Initialization, Parameters & the Dataset

*How weights start, how to count, freeze and inspect them — and the two-method contract that connects your models to real data.*

**PyTorch from Ground Up · Volume I · Chapters 22–23** — companion notebook.

**How to use it:** run the cells from top to bottom. Before each code cell, *predict* what it will print — then run it and compare with the expected output (click ▸ to open it). Every exercise from the lecture is at the end.

By the end you will be able to:
- Explain why initial weights must be small and random, and what happens outside the narrow band.
- Pick an initialization scheme and apply it with `.apply()`.
- Count parameters in total and by name.
- Freeze and unfreeze parameters — and rebuild the optimizer afterwards.
- Inspect parameter statistics to spot dead layers and NaNs.
- State the two-method Dataset contract and implement it for tensors, CSV files and image folders.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
print('PyTorch', torch.__version__)

## 01 · Every parameter starts somewhere

Before step one, every parameter already has a value. That start can decide whether a model **learns at all**.

<sub>Source: lecture slides 3–7</sub>

**`layer_init.py`** — predict the output, then run.

In [ ]:
layer = nn.Linear(4, 3)
print(layer.weight)
print(layer.bias)

<details><summary>▸ Expected output</summary>

```
Parameter containing:
tensor([[ 0.3643, -0.3121,  0.1044,  0.2250],
        [-0.1818,  0.3803,  0.0258,  0.2715],
        [ 0.4339, -0.2264,  0.2610,  0.1007]])
Parameter containing:
tensor([-0.1654,  0.3119,  0.1500])
(random values — yours will differ)
```
</details>

🧪 *Interactive lab in the app:* **The narrow band** — try it there to see this idea move.

## 02 · The schemes that matter, and .apply()

To override the default: write an init function, then **`.apply()` it to the whole model**. `.apply()` visits every sub-module recursively and calls your function on each; the `isinstance` check skips layers without weights, like `ReLU`.

<sub>Source: lecture slides 8–15</sub>

**`init.py`** — predict the output, then run.

In [ ]:
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

model = nn.Sequential(
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)
model.apply(init_weights)
print("Initialized")

<details><summary>▸ Expected output</summary>

```
Initialized
```
</details>

> ⚠️ **Watch out — The trailing underscore means in place**  
> Every `nn.init` function ends with an underscore — `kaiming_normal_`, `zeros_` — and modifies the tensor in place.
>
> Do not assign the result. Write `nn.init.zeros_(m.bias)`, not `m.bias = nn.init.zeros_(m.bias)`.

## 03 · Counting parameters

Count your parameters — to **estimate memory, compare architectures and debug**.

<sub>Source: lecture slides 16–19</sub>

**`count_params.py`** — predict the output, then run.

In [ ]:
total = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total:,}")

for name, p in model.named_parameters():
    print(f"{name:20s} {str(p.shape):18s} {p.numel():>7,}")

<details><summary>▸ Expected output</summary>

```
Total parameters: 101,770
0.weight             torch.Size([128, 784])  100,352
0.bias               torch.Size([128])           128
2.weight             torch.Size([10, 128])     1,280
2.bias               torch.Size([10])             10
```
</details>

## 04 · Freezing and unfreezing

Sometimes parameters should **stop learning**: a pretrained backbone, a tuned early layer. `requires_grad = False` removes a parameter from the gradient computation entirely.

<sub>Source: lecture slides 20–24</sub>

**`freeze.py`** — predict the output, then run.

In [ ]:
# Freeze the first layer
for p in model[0].parameters():
    p.requires_grad = False
# Verify: count trainable vs total
trainable = sum(p.numel() for p in model.parameters()
                if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,}")

<details><summary>▸ Expected output</summary>

```
Trainable: 1,290 / 101,770
```
</details>

🧪 *Interactive lab in the app:* **Freeze, then recount** — try it there to see this idea move.

> ⚠️ **Watch out — Freezing changes what the optimizer should hold**  
> After freezing or unfreezing, **recreate the optimizer** — or at minimum pass it only the trainable parameters.
>
> The clean pattern: `optimizer = Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)`

## 05 · Inspecting statistics



<sub>Source: lecture slides 25–26</sub>

**`stats.py`** — predict the output, then run.

In [ ]:
for name, p in model.named_parameters():
    print(f"{name:20s} mean={p.data.mean():.4f} "
          f"std={p.data.std():.4f} "
          f"min={p.data.min():.4f} "
          f"max={p.data.max():.4f}")

## 06 · The Dataset contract: two methods

Part IV: from hand-typed tensors to **real data**. A Dataset wraps any source behind one simple interface. CSV files, image folders, databases — and once wrapped, batching, shuffling and parallel loading come for free.

<sub>Source: lecture slides 29–34</sub>

🧪 *Interactive lab in the app:* **len(ds) and ds[i]** — try it there to see this idea move.

**`tensor_dataset.py`** — predict the output, then run.

In [ ]:
from torch.utils.data import TensorDataset
x = torch.randn(100, 5)          # 100 samples, 5 features
y = torch.randint(0, 3, (100,))  # 100 labels (3 classes)
ds = TensorDataset(x, y)
print(len(ds))                   # 100
print(ds[0])                     # (tensor([...]), tensor(2))
print(ds[0][0].shape)            # torch.Size([5])

<details><summary>▸ Expected output</summary>

```
100
(tensor([ 1.5410, -0.2934, -2.1788,  0.5684, -1.0845]), tensor(1))
torch.Size([5])
(random values — yours will differ)
```
</details>

## 07 · Custom Datasets: tensors, CSV, and lazy images

Small file? **Read once, index instantly.** Too big for memory? Store paths and **load lazily** in `__getitem__`. The lazy pattern is how every major image dataset works.

<sub>Source: lecture slides 35–46</sub>

**`house_dataset.py`** — predict the output, then run.

In [ ]:
from torch.utils.data import Dataset

class HouseDataset(Dataset):
    def __init__(self, features, prices):
        self.features = features   # (N, D) tensor
        self.prices   = prices     # (N,) tensor
    def __len__(self):
        return len(self.features)
    def __getitem__(self, idx):
        return self.features[idx], self.prices[idx]

feats = torch.randn(200, 3)      # 200 houses, 3 features
prices = torch.randn(200)        # 200 prices
ds = HouseDataset(feats, prices)
print(len(ds))                   # 200
print(ds[0][0].shape)            # (3,) — one house

<details><summary>▸ Expected output</summary>

```
200
torch.Size([3])
```
</details>

**`csv_dataset.py`** — predict the output, then run.

In [ ]:
import csv

class CSVDataset(Dataset):
    def __init__(self, filepath):
        with open(filepath) as f:
            reader = csv.reader(f)
            next(reader)              # skip header
            rows = [row for row in reader]
        self.features = torch.tensor(
            [[float(v) for v in row[:-1]]
             for row in rows])
        self.labels = torch.tensor(
            [int(row[-1]) for row in rows])
    def __len__(self):
        return len(self.features)
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

**`image_dataset.py`** — predict the output, then run.

In [ ]:
from pathlib import Path
# from PIL import Image          # uncomment with real images

class ImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.paths = sorted(
            Path(root_dir).glob("*/*.jpg"))
        self.classes = sorted(
            {p.parent.name for p in self.paths})
        self.class_to_idx = {
            c: i for i, c in enumerate(self.classes)}
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        path = self.paths[idx]
        # img = Image.open(path).convert("RGB")
        img = torch.randn(3, 32, 32)   # placeholder
        label = self.class_to_idx[path.parent.name]
        if self.transform:
            img = self.transform(img)
        return img, label

> ⚠️ **Watch out — Return tensors**  
> `__getitem__` should return **tensors**, not Python lists or NumPy arrays.
>
> The DataLoader expects tensors so it can stack them into batches. If you return raw Python objects, you will get cryptic collation errors later.

## 08 · Built-in datasets and what the DataLoader expects

Built-in datasets follow the **same two-method contract** — pass them straight to a DataLoader. `img` here is still a PIL Image; the transform turns it into a tensor next lecture.

<sub>Source: lecture slides 47–49</sub>

**`mnist.py`** — predict the output, then run.

In [ ]:
from torchvision import datasets
# Downloads MNIST to ./data on first run
mnist = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
)
print(len(mnist))                # 60000
img, label = mnist[0]
print(type(img), label)          # PIL Image, 5

<details><summary>▸ Expected output</summary>

```
60000
<class 'PIL.Image.Image'> 5
```
</details>

## The rule — worth remembering

**Initialization & parameters — six rules**

1. PyTorch defaults to Kaiming initialization for Linear layers — correct for ReLU networks.
2. Use `model.apply(init_fn)` to override. All init functions are in-place (trailing underscore).
3. Count parameters: `sum(p.numel() for p in model.parameters())`.
4. Freeze with `p.requires_grad = False`. Unfreeze by setting it back to `True`.
5. After freezing/unfreezing, recreate the optimizer with only trainable parameters.
6. Inspect stats (mean, std, min, max) to catch dead layers, exploding weights, or NaN.

**Dataset — six rules**

1. A Dataset implements two methods: `__len__` (how many samples) and `__getitem__` (one sample by index).
2. `TensorDataset` wraps existing tensors — no class needed.
3. For custom data: inherit from `Dataset`, load data in `__init__`, return one sample in `__getitem__`.
4. For large data (images): store paths in init, load lazily in getitem.
5. The DataLoader stacks them into batches — `__getitem__` should return tensors.
6. Built-in datasets (`torchvision.datasets`) follow the same contract and work out of the box.

## Pause and try

Work each exercise in the empty cell below it. Open the worked answer only after you have tried.

### Exercise 01 · Try it — Xavier everywhere

Create a model with three Linear layers and ReLU activations. Apply Xavier initialization to all weights and zero initialization to all biases. Print one weight tensor before and after to confirm it changed.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

Write `def init(m): if isinstance(m, nn.Linear): nn.init.xavier_normal_(m.weight); nn.init.zeros_(m.bias)` and call `model.apply(init)`. Printing `model[0].weight[0, :5]` before and after shows different values, and every bias is now exactly 0.
</details>

### Exercise 02 · Try it — freeze a block of the Classifier

Build the Classifier from Lecture 12 (784 → 128 → 64 → 10 with BatchNorm, ReLU, Dropout). Count total parameters, then freeze the first block and recount trainable parameters. Print the stats of each weight tensor.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

Total: (100,352 + 128) + 256 [BatchNorm γ, β] + (8,192 + 64) + 128 + (640 + 10) = **109,770**. Freezing the first Linear and its BatchNorm removes 100,736, leaving **9,034** trainable.
</details>

### Exercise 03 · Ch 22 · Ex 1

Create an `nn.Linear(100, 50)`. Print the weight’s mean and std. Apply Xavier normal init and print again — how did the distribution change?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

The mean stays near 0 in both cases. The default (Kaiming uniform) gives std ≈ 0.058; Xavier normal sets std = √(2 / (100 + 50)) ≈ 0.115 — about twice as wide.
</details>

### Exercise 04 · Ch 22 · Ex 4

Freeze every layer except the last. Print trainable vs total. Pass a batch through, call `backward()`, and check `.grad` on a frozen parameter — is it `None`?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

Yes — a frozen parameter’s `.grad` stays `None`, because it was removed from the gradient computation entirely.
</details>

### Exercise 05 · Ch 22 · Ex 5

Initialize a Linear layer’s weight with all zeros and train a model containing it for 100 epochs. Does it learn? Why or why not?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

Poorly or not at all: with zero weights every neuron in the layer produces the same output and receives the same gradient, so they stay identical — and behind a ReLU a zero pre-activation passes no gradient. Symmetry is never broken.
</details>

### Exercise 06 · Ch 23 · Ex 2

Write a `SineDataset` with `n_points` samples of x from 0 to 2π and y = sin(x). Verify by indexing into it.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

In `__init__`: `self.x = torch.linspace(0, 2 * torch.pi, n_points).unsqueeze(1); self.y = torch.sin(self.x)`. `__len__` returns `len(self.x)`; `__getitem__` returns `self.x[idx], self.y[idx]`. `ds[0]` → `(tensor([0.]), tensor([0.]))`.
</details>

### Exercise 07 · Ch 23 · Ex 5 and “wrap NumPy arrays”

Make `__getitem__` return a Python list instead of a tensor and wrap it in `DataLoader(ds, batch_size=4)`. What happens? Then write a Dataset that wraps two NumPy arrays correctly.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

The default collation cannot stack the samples into one batch tensor the way you intended (lists are collated element by element, or it errors on ragged data). The fix is to convert once in `__init__`: `self.x = torch.from_numpy(features).float(); self.y = torch.from_numpy(labels).long()`.
</details>

### Interview-style

(a) Why is it bad to initialize all weights to zero? And what initialization would you use for a network with ReLU activations? — three sentences.

_Write your answer here before opening the model answer._

<details><summary>▸ Model answer</summary>

With all-zero weights every neuron in a layer computes the same output and receives the same gradient, so they update identically and never learn different features. Random initialization breaks that symmetry. For ReLU networks use Kaiming (He) initialization, which scales the random values so signals neither shrink nor explode — and which is already PyTorch’s default for Linear layers.
</details>

### Interview-style

(b) What two methods must a PyTorch Dataset implement, and why does the DataLoader need them? — three sentences. When would you load data lazily in `__getitem__` instead of all at once in `__init__`? Give an example.

_Write your answer here before opening the model answer._

<details><summary>▸ Model answer</summary>

`__len__`, which returns the number of samples, and `__getitem__`, which returns the sample at a given index. The DataLoader uses the length to plan an epoch and the indexing to fetch individual samples, which it then shuffles, stacks into batches and loads in parallel. Load lazily when the data doesn’t fit in memory — an image dataset stores only file paths in `__init__` and opens each image on demand.
</details>